In [ ]:
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.4/502.4 kB 13.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

from collections import defaultdict

In [ ]:
data = pd.read_csv("data (1).csv")

data.head()

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667.0,0.211,0.0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10.0,0.665,-20.096,1.0,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4.0,1921,0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,180533.0,0.341,0.0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7.0,0.160,-12.441,1.0,Clancy Lowered the Boom,5.0,1921,0.4150,60.936
2,0.0394,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062.0,0.166,0.0,1o6I8BglA6ylDMrIELygv1,0.913000,3.0,0.101,-14.850,1.0,Gati Bali,5.0,1921,0.0339,110.339
3,0.1650,1921,0.967,['Frank Parker'],0.275,210000.0,0.309,0.0,3ftBPsC5vPBKxYSee08FDH,0.000028,5.0,0.381,-9.316,1.0,Danny Boy,3.0,1921,0.0354,100.109
4,0.2530,1921,0.957,['Phil Regan'],0.418,166693.0,0.193,0.0,4d6HGyGT8e121BsdKmw9v6,0.000002,3.0,0.229,-10.096,1.0,When Irish Eyes Are Smiling,2.0,1921,0.0380,101.665


In [ ]:

CLIENT_ID = "THE_CLIENT_ID"
CLIENT_SECRET = "THE_CLIENT_SECRET""

auth_manager = SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

sp = spotipy.Spotify(
    auth_manager=auth_manager
)

print("Spotify Connected Successfully!")

Spotify Connected Successfully!


In [ ]:
number_cols = [
    'valence',
    'year',
    'acousticness',
    'danceability',
    'duration_ms',
    'energy',
    'explicit',
    'instrumentalness',
    'key',
    'liveness',
    'loudness',
    'mode',
    'popularity',
    'speechiness',
    'tempo'
]

# Keep only rows without missing numerical values
data = data.dropna(subset=number_cols).reset_index(drop=True)

scaler = StandardScaler()

scaled_data = scaler.fit_transform(data[number_cols])

In [ ]:
scaler = StandardScaler()

scaled_data = scaler.fit_transform(data[number_cols])

scaled_data = pd.DataFrame(
    scaled_data,
    columns=number_cols
)

In [ ]:
def find_song(name, year):

    song = data[
        (data['name'].str.lower() == name.lower())
        &
        (data['year'] == year)
    ]

    if len(song):

        return song.iloc[0]

    return None

In [ ]:
def get_song_data(song):

    song_data = find_song(
        song['name'],
        song['year']
    )

    if song_data is not None:

        return song_data

    try:

        results = sp.search(
            q=f'track:{song["name"]}',
            limit=1,
            type='track'
        )

        items = results['tracks']['items']

        if len(items)==0:

            return None

        track = items[0]

        return {
            'name':track['name'],
            'year':int(track['album']['release_date'][:4])
        }

    except:

        return None

In [ ]:
def get_mean_vector(song_list):

    song_vectors = []

    for song in song_list:

        song_data = get_song_data(song)

        if song_data is None:

            continue

        if isinstance(song_data, dict):

            song_data = find_song(
                song_data['name'],
                song_data['year']
            )

        if song_data is None:

            continue

        song_vectors.append(
            song_data[number_cols].values
        )

    song_matrix = np.array(song_vectors)

    return np.mean(
        song_matrix,
        axis=0
    )

In [ ]:
def flatten_dict_list(dict_list):

    flattened = defaultdict(list)

    for dictionary in dict_list:

        for key,value in dictionary.items():

            flattened[key].append(value)

    return flattened

In [ ]:
def recommend_songs(song_list,
                    n_songs=10):

    metadata_cols = [
        'name',
        'year',
        'artists'
    ]

    song_center = get_mean_vector(song_list)

    scaled_center = scaler.transform(
        pd.DataFrame(
            [song_center],
            columns=number_cols
        )
    )

    similarities = cosine_similarity(
        scaled_center,
        scaled_data
    )

    recommendations = similarities.argsort()[0][-n_songs:][::-1]

    rec_songs = data.iloc[
        recommendations
    ][metadata_cols]

    return rec_songs.reset_index(drop=True)

In [ ]:
recommend_songs(
    [
        {
            'name':'Shape of You',
            'year':2017
        }
    ]
)

,name,year,artists
0,Shape of You,2017,['Ed Sheeran']
1,Because Of You,2007,['Ne-Yo']
2,Nunca Es Suficiente,2018,"['Los Ángeles Azules', 'Natalia Lafourcade']"
3,Métele Sazón,2003,"['Luny Tunes', 'Noriega', 'Tego Calderon']"
4,Besar Tu Piel,1997,['Rayito Colombiano']
5,Baby I'm Yours,2010,"['Breakbot', 'Irfane']"
6,Attention,2018,['Charlie Puth']
7,Bust Your Windows,2008,['Jazmine Sullivan']
8,Más Que Tu Amigo,2003,['Marco Antonio Solís']
9,Love You Like A Love Song,2011,['Selena Gomez & The Scene']


In [ ]:
recommend_songs(
    [
        {
            'name':'Blinding Lights',
            'year':2020
        }
    ]
)

,name,year,artists
0,Blinding Lights,2020,['The Weeknd']
1,Secrets,2009,['OneRepublic']
2,They Don't Know About Us,2012,['One Direction']
3,Thunder,2017,['Imagine Dragons']
4,Forever After All,2020,['Luke Combs']
5,Getaway Car,2017,['Taylor Swift']
6,Fireflies,2009,['Owl City']
7,Versace on the Floor,2016,['Bruno Mars']
8,Drunk Me,2018,['Mitchell Tenpenny']
9,Mind Over Matter,2014,['Young the Giant']
